# SHAP MAPS

In [1]:
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'
model_dir = future_dir + "Models"
shap_dir = future_dir + "SHAP"

In [ ]:
# Read in HUC12 centroids (6s)
import geopandas as gpd
huc12_centroids_path = 'D:/ArcGIS/Boundaries/HUC12_catchments/HUC12_centroids.shp'
huc12_p = gpd.read_file(huc12_centroids_path)

c:\Users\MPennino\anaconda3\envs\shapmap_env\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


In [ ]:
# Read in COMID centroids (12s)
import geopandas as gpd
sp_dir = "D:/Data/Spatial/NHDPlusV2_Centroids/"

cats_centroids_path = sp_dir + "NHDPlusV2_Cat_All_centroid_point.shp"
cats_p = gpd.read_file(cats_centroids_path)



In [10]:
cats_p.columns

Index(['COMID', 'SOURCEFC', 'geometry'], dtype='str')

In [ ]:
# Change projection (1s)
cats_p = cats_p.to_crs("EPSG:5070")


In [12]:
epsg_code = cats_p.crs.to_epsg()
print(epsg_code)

5070


In [17]:
# Import SHAP Values Dataset

import pickle
import shap
import torch

filename2 = '/torch_SHAP_Values_kfold_COMID_All_Observed_sw.pkl'

with open(shap_dir + filename2, "rb") as f:
    shap_values = pickle.load(f)

type(shap_values)

shap._explanation.Explanation

In [27]:
# Import SHAP values DataFrame
import pandas as pd
#import pyarrow as pa
#import pyarrow.parquet as pq
dataset = "/torch_SHAP_Values_kfold_DF_COMID_All_Observed_sw.parquet"
#df_shap = pd.read_parquet(shap_dir+dataset) # Read a single Parquet file

import pyarrow.parquet as pq

df_shap = pq.read_table(shap_dir+dataset).to_pandas()

In [28]:
df_shap.head()

,COMID,PopDen2010Ws,PctForest2019Ws,PctCrop2019Ws,precip9120ws,tmean9120ws,BFIWs,permws,RockNWs,N_TW2012Ws,N_Surp_kgsqkm_2017ws,ElevWs,Fe2O3Ws,NHDslope_Pct_Ws
0,12558,1.083973,-0.127625,-0.232869,1.779821,-1.484759,1.452009,0.042613,1.612080,-0.183911,2.546797,0.174274,-0.019606,-0.114608
1,12564,1.370863,-1.702490,-2.251283,4.546871,-1.316279,2.181968,0.067828,-2.339952,-0.120063,-6.593807,7.460185,0.083605,-0.660099
2,12606,3.161424,1.412491,-0.910008,3.142874,-1.865382,3.937159,1.928581,-0.333684,-0.132065,3.377100,-7.475146,-0.305945,-0.390846
3,12678,-5.258402,0.243095,0.369965,5.691373,-1.609480,-0.029447,-0.263404,0.985291,-0.173745,2.640598,-5.860209,0.010729,-0.008226
4,12712,2.137773,0.648742,-0.797457,3.016529,-1.933363,3.414159,1.811792,-0.022450,-0.134768,3.881362,-4.598128,-0.196464,-0.433076


In [29]:
# Merge Prediction results with Shapefile
shap_s = cats_p.merge(df_shap, on="COMID", how="left")
shap_s.shape, shap_s.columns, type(shap_s)

((2647466, 16),
 Index(['COMID', 'SOURCEFC', 'geometry', 'PopDen2010Ws', 'PctForest2019Ws',
        'PctCrop2019Ws', 'precip9120ws', 'tmean9120ws', 'BFIWs', 'permws',
        'RockNWs', 'N_TW2012Ws', 'N_Surp_kgsqkm_2017ws', 'ElevWs', 'Fe2O3Ws',
        'NHDslope_Pct_Ws'],
       dtype='str'),
 geopandas.geodataframe.GeoDataFrame)

In [30]:
epsg_code = shap_s.crs.to_epsg()
print(epsg_code)

5070
